In [3]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

"""
과제: 서울시 OpenAPI(지하철 승하차 통계) 데이터 수집

참고 사이트
- 서울 열린데이터광장: https://data.seoul.go.kr
- 지하철 api 사이트: https://data.seoul.go.kr/dataList/OA-12912/S/1/datasetView.do

목표
- 서울시 OpenAPI를 호출하여 특정 날짜의 지하철 승하차 데이터를 수집한다.
- XML 응답을 파싱하여 필요한 컬럼만 추출한다.
- DataFrame으로 변환한 뒤 결과를 확인한다.
"""


'\n과제: 서울시 OpenAPI(지하철 승하차 통계) 데이터 수집\n\n참고 사이트\n- 서울 열린데이터광장: https://data.seoul.go.kr\n- 지하철 api 사이트: https://data.seoul.go.kr/dataList/OA-12912/S/1/datasetView.do\n\n목표\n- 서울시 OpenAPI를 호출하여 특정 날짜의 지하철 승하차 데이터를 수집한다.\n- XML 응답을 파싱하여 필요한 컬럼만 추출한다.\n- DataFrame으로 변환한 뒤 결과를 확인한다.\n'

In [4]:
# 문제 1
# 본인 인증키로 바꾸시오.
API_KEY = "625459744474726537344a72466f73"

# 문제 2
# 조회할 날짜(YYYYMMDD)로 바꾸시오.
DATE = "202604010"

# 서울시 OpenAPI 기본 URL
BASE_URL = "http://openapi.seoul.go.kr:8088/"

# 지하철 승하차 통계 API
SERVICE_NAME = "CardSubwayStatsNew"
START_INDEX = 1
END_INDEX = 50

In [5]:
# 문제 3
# 아래 형식에 맞게 요청 URL을 완성하시오.
# 형식: {BASE_URL}{API_KEY}/xml/{SERVICE_NAME}/{START_INDEX}/{END_INDEX}/{DATE}
url = f"{BASE_URL}{API_KEY}/xml/{SERVICE_NAME}/{START_INDEX}/{END_INDEX}/{DATE}"

print("[1] 요청 URL:")
print(url)

[1] 요청 URL:
http://openapi.seoul.go.kr:8088/625459744474726537344a72466f73/xml/CardSubwayStatsNew/1/50/202604010


In [6]:
# 문제 4
# requests 라이브러리를 사용하여 GET 요청을 보내시오.
response = requests.get(url,)

print("\n[2] HTTP 상태 코드:")
print(response.status_code)


[2] HTTP 상태 코드:
200


In [7]:
# 문제 5
# 상태 코드가 200이 아니면 에러를 발생시키시오.
if response.status_code != 200:
    raise RuntimeError("API 호출 실패: 상태코드를 확인하세요.")


In [8]:
# 문제 6
# XML 파싱을 수행하시오. (응답 본문을 사용)
root = ET.fromstring(response.text)

In [9]:
print(response.text)

<?xml version="1.0" encoding="UTF-8"?>
<RESULT>
<CODE>INFO-200</CODE>
<MESSAGE>해당하는 데이터가 없습니다.</MESSAGE>
</RESULT>



In [10]:
# 문제 7
# XML에서 row 태그 목록을 찾아 rows 변수에 저장하시오.
rows = root.findall('.//row')
print("\n[3] row 개수:", len(rows))


[3] row 개수: 0


In [11]:
# 문제 8
# rows를 순회하면서 필요한 컬럼을 추출해 data 리스트를 완성하시오.
# - 날짜: USE_YMD
# - 노선번호(노선명): SBWY_ROUT_LN_NM
# - 역명: SBWY_STNS_NM
# - 승차인원: GTON_TNOPE (정수 변환, 숫자가 아니면 0)
data = []

for row in rows:
    use_ymd = row.findtext("USE_YMD", default="")
    line_nm = row.findtext("SBWY_ROUT_LN_NM", default="")
    station_nm = row.findtext("SBWY_STNS_NM", default="")
    ride = row.findtext("GTON_TNOPE", default="0")

    data.append({
        "날짜": use_ymd,
        "노선번호": line_nm,
        "역명": station_nm,
        "승차인원": int(float(ride)) if ride.replace('.','',1).isdigit() else 0,
    })

In [12]:
# 문제 9
# data를 DataFrame으로 변환하시오.
df = pd.DataFrame(data)

print("\n[4] 결과 미리보기:")
print(df.head(10))

print("\n[5] 컬럼 확인:")
print(df.columns.tolist())


[4] 결과 미리보기:
Empty DataFrame
Columns: []
Index: []

[5] 컬럼 확인:
[]
